# Model Results Comparison

Purpose: provide one notebook for the final comparison table across the ResNet18 and ConvNeXtV2 runs owned in this cleanup pass.

This notebook uses `reports/tables/model_results_master.csv` and `reports/tables/model_results_master.json` as the source of truth.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

ROOT = Path.cwd()
while not (ROOT / 'pyproject.toml').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

master_path = ROOT / 'reports' / 'tables' / 'model_results_master.csv'
summary_path = ROOT / 'reports' / 'tables' / 'model_results_master.json'
master = pd.read_csv(master_path)
summary = json.loads(summary_path.read_text(encoding='utf-8'))

metric_cols = [
    'model_name', 'run_group', 'pretrained', 'seed', 'split',
    'accuracy', 'precision_macro', 'recall_macro', 'f1_macro',
    'best_epoch', 'epochs_trained', 'recommendation', 'notes'
]

In [ ]:
master.sort_values(['model_family', 'run_group', 'seed'])[metric_cols]

In [ ]:
summary_rows = []
for run_group, values in summary['summary_by_run_group'].items():
    row = {'run_group': run_group}
    row.update(values)
    summary_rows.append(row)
summary_df = pd.DataFrame(summary_rows)
summary_df.sort_values('mean_accuracy', ascending=False)

In [ ]:
plot_df = summary_df.dropna(subset=['mean_accuracy']).sort_values('mean_accuracy')
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(plot_df['run_group'], plot_df['mean_accuracy'])
ax.set_xlabel('Mean accuracy')
ax.set_xlim(0, 1.05)
ax.set_title('Mean validation accuracy by run group')
for idx, value in enumerate(plot_df['mean_accuracy']):
    ax.text(value + 0.005, idx, f'{value:.4f}', va='center')
plt.tight_layout()
plt.show()

## Deployment-Oriented Interpretation

For deployment, use a reliable and simple model first. ResNet18 transfer learning remains the safest first ONNX export candidate because it is smaller and easier to explain than ConvNeXt while still performing strongly.

ConvNeXt scratch and pretrained results remain useful for comparison and discussion, especially around pretrained versus non-pretrained behavior.